# LangChain: Models, Prompts and Output Parsers


## Outline

 * Direct API calls to OpenAI
 * API calls through LangChain:
   * Prompts
   * Models
   * Output parsers

## Get your [OpenAI API Key](https://platform.openai.com/account/api-keys)

In [73]:
!pip install python-dotenv
!pip install openai

In [74]:
import os
from openai import OpenAI

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url = "https://openrouter.ai/api/v1"
)

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [75]:
llm_model = "google/gemma-4-26b-a4b-it:free"

## Chat API : OpenAI

Let's start with a direct API calls to OpenAI.

In [76]:
def get_completion(prompt, model=llm_model):
    response = client.responses.create(
        model=model,
        input=prompt,
        temperature=0, 
    )
    return response.output_text


In [77]:
get_completion("What is 1+1?")

'1 + 1 = 2'

In [78]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse,\
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [79]:
style = """American English \
in a calm and respectful tone
"""

In [80]:
prompt = f"""Translate the text \
that is delimited by triple backticks 
into a style that is {style}.
text: ```{customer_email}```
"""

print(prompt)

Translate the text that is delimited by triple backticks 
into a style that is American English in a calm and respectful tone
.
text: ```
Arrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse,the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!
```



In [81]:
response = get_completion(prompt)

In [82]:
response

'I am quite frustrated because my blender lid popped off, splashing my smoothie all over the kitchen walls. To make matters worse, the warranty does not cover the cost of cleaning up the mess. I would appreciate your assistance with this matter as soon as possible.'

## Chat API : LangChain

Let's try how we can do the same using LangChain.

In [83]:
!pip install --upgrade langchain

### Model

In [84]:
%pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [85]:
from langchain_openai import ChatOpenAI

In [106]:
# To control the randomness and creativity of the generated
# text by an LLM, use temperature = 0.0
chat = ChatOpenAI(
    model=llm_model,
    temperature=0.0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)
chat

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.1'}}, output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x000001DC194AE140>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001DC194AE470>, root_client=<openai.OpenAI object at 0x000001DC194ADF20>, root_async_client=<openai.AsyncOpenAI object at 0x000001DC194AE250>, model_name='google/gemma-4-26b-a4b-it:free', temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None, stream_chunk_timeout=120.0)

### Prompt template

In [87]:
template_string = """Translate the text \
that is delimited by triple backticks \
into a style that is {style}. \
text: ```{text}```
"""

In [88]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(template_string)


In [89]:
prompt_template.messages[0].prompt

PromptTemplate(input_variables=['style', 'text'], input_types={}, partial_variables={}, template='Translate the text that is delimited by triple backticks into a style that is {style}. text: ```{text}```\n')

In [90]:
prompt_template.messages[0].prompt.input_variables

['style', 'text']

In [91]:
customer_style = """American English \
in a calm and respectful tone
"""

In [92]:
customer_email = """
Arrr, I be fuming that me blender lid \
flew off and splattered me kitchen walls \
with smoothie! And to make matters worse, \
the warranty don't cover the cost of \
cleaning up me kitchen. I need yer help \
right now, matey!
"""

In [93]:
customer_messages = prompt_template.format_messages(
                    style=customer_style,
                    text=customer_email)

In [94]:
print(type(customer_messages))
print(type(customer_messages[0]))

<class 'list'>
<class 'langchain_core.messages.human.HumanMessage'>


In [95]:
print(customer_messages[0])

content="Translate the text that is delimited by triple backticks into a style that is American English in a calm and respectful tone\n. text: ```\nArrr, I be fuming that me blender lid flew off and splattered me kitchen walls with smoothie! And to make matters worse, the warranty don't cover the cost of cleaning up me kitchen. I need yer help right now, matey!\n```\n" additional_kwargs={} response_metadata={}


In [107]:
# Call the LLM to translate to the style of the customer message
customer_response = chat.invoke(customer_messages)

In [108]:
print(customer_response.content)

I am quite frustrated because my blender lid popped off, splashing my smoothie all over the kitchen walls. To make matters worse, the warranty does not cover the cost of cleaning up the mess. I would appreciate your assistance with this matter as soon as possible.


In [109]:
service_reply = """Hey there customer, \
the warranty does not cover \
cleaning expenses for your kitchen \
because it's your fault that \
you misused your blender \
by forgetting to put the lid on before \
starting the blender. \
Tough luck! See ya!
"""

In [110]:
service_style_pirate = """\
a polite tone \
that speaks in English Pirate\
"""

In [111]:
service_messages = prompt_template.format_messages(
    style=service_style_pirate,
    text=service_reply)

print(service_messages[0].content)

Translate the text that is delimited by triple backticks into a style that is a polite tone that speaks in English Pirate. text: ```Hey there customer, the warranty does not cover cleaning expenses for your kitchen because it's your fault that you misused your blender by forgetting to put the lid on before starting the blender. Tough luck! See ya!
```



In [113]:
service_response = chat.invoke(service_messages)
print(service_response.content)

```Ahoy there, valued matey! We be reachin' out to ye regarding yer claim. Regrettably, the warranty don't cover the cost o' cleanin' yer galley, for it appears the mishap was caused by a lapse in navigation—specifically, leavin' the lid off yer blender before settin' sail! 'Tis a heavy blow, to be sure, and we be sorry we cannot aid ye this time. Fair winds to ye, and may yer future voyages be less messy!```


## Output Parsers

Let's start with defining how we would like the LLM output to look like:

In [114]:
{
  "gift": False,
  "delivery_days": 5,
  "price_value": "pretty affordable!"
}

{'gift': False, 'delivery_days': 5, 'price_value': 'pretty affordable!'}

In [115]:
customer_review = """\
This leaf blower is pretty amazing.  It has four settings:\
candle blower, gentle breeze, windy city, and tornado. \
It arrived in two days, just in time for my wife's \
anniversary present. \
I think my wife liked it so much she was speechless. \
So far I've been the only one using it, and I've been \
using it every other morning to clear the leaves on our lawn. \
It's slightly more expensive than the other leaf blowers \
out there, but I think it's worth it for the extra features.
"""

review_template = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product \
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

Format the output as JSON with the following keys:
gift
delivery_days
price_value

text: {text}
"""

In [116]:
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(review_template)
print(prompt_template)

input_variables=['text'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='For the following text, extract the following information:\n\ngift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.\n\ndelivery_days: How many days did it take for the product to arrive? If this information is not found, output -1.\n\nprice_value: Extract any sentences about the value or price,and output them as a comma separated Python list.\n\nFormat the output as JSON with the following keys:\ngift\ndelivery_days\nprice_value\n\ntext: {text}\n'), additional_kwargs={})]


In [119]:
messages = prompt_template.format_messages(text=customer_review)
chat = ChatOpenAI(
    model=llm_model,
    temperature=0.0,
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)
response = chat.invoke(messages)
print(response.content)

```json
{
  "gift": true,
  "delivery_days": 2,
  "price_value": [
    "It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."
  ]
}
```


In [120]:
type(response.content)

str

In [121]:
# You will get an error by running this line of code 
# because'gift' is not a dictionary
# 'gift' is a string
response.content.get('gift')

AttributeError: 'str' object has no attribute 'get'

### Parse the LLM output string into a Python dictionary

In [123]:
from pydantic import BaseModel,Field

In [124]:
class ProductReview(BaseModel):
    gift:bool = Field(
        description="Was the item purchased as a gift for someone else? "
                    "True if yes, False if not or unknown."
    )



    delivery_days:int = Field(
        description="How many days did it take for the product to arrive? "
                    "If this information is not found, output -1"
    )

    price_value:list[str] = Field(
        description="Extract any sentences about the value or price," \
        "and output them as a comma separated Python list"
    )


In [ ]:
structured_model =  chat.with_structured_output(ProductReview)

In [129]:
result = structured_model.invoke(customer_review)

In [130]:
print(result)

gift=True delivery_days=2 price_value=["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


In [131]:
review_template_2 = """\
For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? \
Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the product\
to arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,\
and output them as a comma separated Python list.

text: {text}

"""

prompt = ChatPromptTemplate.from_template(template=review_template_2)

messages = prompt.format_messages(text=customer_review 
                                )

In [132]:
print(messages[0].content)

For the following text, extract the following information:

gift: Was the item purchased as a gift for someone else? Answer True if yes, False if not or unknown.

delivery_days: How many days did it take for the productto arrive? If this information is not found, output -1.

price_value: Extract any sentences about the value or price,and output them as a comma separated Python list.

text: This leaf blower is pretty amazing.  It has four settings:candle blower, gentle breeze, windy city, and tornado. It arrived in two days, just in time for my wife's anniversary present. I think my wife liked it so much she was speechless. So far I've been the only one using it, and I've been using it every other morning to clear the leaves on our lawn. It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features.





In [133]:
response = chat.invoke(messages)

In [134]:
print(response.content)

gift: True
delivery_days: 2
price_value: ["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


In [139]:
#new langchain update method
result = structured_model.invoke(messages)
print(result)

gift=True delivery_days=2 price_value=["It's slightly more expensive than the other leaf blowers out there, but I think it's worth it for the extra features."]


In [ ]:
output_dict = output_parser.parse(response.content)

In [ ]:
output_dict

In [138]:
type(response)

langchain_core.messages.ai.AIMessage

In [ ]:
output_dict.get('delivery_days')

Reminder: Download your notebook to you local computer to save your work.

In [140]:
import os
os.getcwd()

'f:\\LangChain'